# Word2Vec (mean pooling) + Logistic Regression

**Task:** Abductive Commonsense Reasoning (ART). Given two observations O1, O2 and two
hypotheses H1, H2, pick the hypothesis that better explains what happened in between.

**Approach of this notebook:** represent every sentence as the *mean* of its pretrained
word2vec vectors and let a logistic regression decide between the two hypotheses.

In [1]:
%pip install -q datasets gensim nltk python-dotenv scikit-learn wandb

Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))  # notebooks/project_1 -> shared pipeline package

import joblib
from sklearn.linear_model import LogisticRegression

from pipeline import tracking
from pipeline.data import DATASET, TEXT_FIELDS, load_splits
from pipeline.evaluation import majority_accuracy
from pipeline.features.static_embeddings import MeanEmbedder, build_features, combine_mean, embed_example, load_vectors
from pipeline.preprocessing import tokenize_split
from pipeline.preprocessing import nltk_alnum as preprocessing  # swap the preprocessing variant here
from pipeline.training import weights_dir

True

In [2]:
tracking.login()

environment loaded from: /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/.env


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: cmedi32 (cmedi32-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
### Weights and Biases Init and Data Download

In [3]:
RUN_NAME = "w2v-mean-logreg"  # W&B run + artifact name, weights in models/project_1/w2v_mean_logreg/weights

config = {
    "dataset": DATASET,
    **preprocessing.CONFIG,  # tokenizer, lowercase, remove_punctuation, remove_stopwords, normalization, truncation
    "embedding": "word2vec-google-news-300",  # gensim downloader model id: 300 dims, ~3M words, 1.7 GB
    "sentence_representation": "mean",
    "classifier": "logistic_regression",
}

run = tracking.init_run(RUN_NAME, config)

In [4]:
train, valid, test = load_splits()

print(train[0])

example observation printed
train: 16801, valid: 1000, test: 1532
{'observation_1': 'A baby panda was lost in the woods.', 'observation_2': 'He was reunited with his parents, who also lived at the zoo.', 'hypothesis_1': 'He was never lost.', 'hypothesis_2': 'All the people from the zoo looked for him.', 'label': 2}


### NLP pipeline

#### Preprocessing decisions

The target model is **pretrained word2vec (`word2vec-google-news-300`)**. Its vocabulary
is fixed and was built by somebody else, so we cannot adapt the model to our tokens — we
have to adapt our tokens to the model.

#### Filtering Tokens

raw text:  He was reunited with his parents, who also lived at the zoo.
tokenized: ['He', 'was', 'reunited', 'with', 'his', 'parents', ',', 'who', 'also', 'lived', 'at', 'the', 'zoo', '.']
filtered:  ['He', 'was', 'reunited', 'with', 'his', 'parents', 'who', 'also', 'lived', 'at', 'the', 'zoo']

In [19]:
# preprocessing decisions above: pipeline/preprocessing/nltk_alnum.py
train_tokens = tokenize_split(train, preprocessing.preprocess)
valid_tokens = tokenize_split(valid, preprocessing.preprocess)
test_tokens = tokenize_split(test, preprocessing.preprocess)

A quick look at one example, before and after preprocessing. This is the check that the
decisions above actually took effect: punctuation is gone, casing and stopwords are
untouched, word forms are unchanged, nothing is cut off.

In [8]:
for field in TEXT_FIELDS:
    print(f"{field}:")
    print(f"  train:    {train[0][field]}")
    print(f"  train_tokens: {train_tokens[0][field]}")

observation_1:
  train:    A baby panda was lost in the woods.
  train_tokens: ['A', 'baby', 'panda', 'was', 'lost', 'in', 'the', 'woods']
observation_2:
  train:    He was reunited with his parents, who also lived at the zoo.
  train_tokens: ['He', 'was', 'reunited', 'with', 'his', 'parents', 'who', 'also', 'lived', 'at', 'the', 'zoo']
hypothesis_1:
  train:    He was never lost.
  train_tokens: ['He', 'was', 'never', 'lost']
hypothesis_2:
  train:    All the people from the zoo looked for him.
  train_tokens: ['All', 'the', 'people', 'from', 'the', 'zoo', 'looked', 'for', 'him']


In [9]:
wv = load_vectors(config["embedding"])

vocabulary: 3000000 words, 300 dimensions


In [10]:
# google news vectors are cased: an unknown token gets a second lookup in lowercase
embed = MeanEmbedder(wv, lowercase_fallback=True)

In [12]:
X_train, y_train = build_features(train_tokens, embed, combine_mean) # → 1200d and predict label 1 vs 2 directly. simpler could also have one feature per hypotheses
X_valid, y_valid = build_features(valid_tokens, embed, combine_mean)

# Decision Remaining: Whether the averaged vectors are L2-normalized before the logistic regression.

classifier = LogisticRegression(max_iter=1000) # simple regression, puts out weight tensor and bias term
classifier.fit(X_train, y_train)

train_accuracy = classifier.score(X_train, y_train)
valid_accuracy = classifier.score(X_valid, y_valid)

print(f"train accuracy: {train_accuracy:.2%}")
print(f"valid accuracy: {valid_accuracy:.2%}")

train accuracy: 54.89%
valid accuracy: 52.70%


In [13]:
WEIGHTS_DIR = weights_dir(RUN_NAME)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(classifier, WEIGHTS_DIR / "logreg.joblib") # output classifiers weights / ai generated, not sure if i need it at the moment

['../../../models/project_1/w2v_mean_logreg/weights/logreg.joblib']

In [14]:
def forward(split):
    vector = embed_example(split[0], embed, combine_mean)
    probabilities = classifier.predict_proba(vector.reshape(1, -1))[0]
    return probabilities

probabilities = forward(valid_tokens)
print(probabilities)

[0.50851972 0.49148028]


In [15]:
run.log({
    **embed.summary(),  # oov/tokens, oov/occurrences, oov/rate, oov/unique_types, oov/empty_sentences
    "accuracy/train": train_accuracy,
    "accuracy/valid": valid_accuracy,
})

In [16]:
X_test, y_test = build_features(test_tokens, embed, combine_mean)

test_accuracy = classifier.score(X_test, y_test)

majority = majority_accuracy(y_test)
oov_rate = embed.summary()["oov/rate"]

In [17]:
predictions = classifier.predict(X_test)

In [18]:
run.summary.update({
    "accuracy/train": train_accuracy,
    "accuracy/valid": valid_accuracy,
    "accuracy/test": test_accuracy,
    "accuracy/majority": majority,
    "oov/rate": oov_rate,
})

run.finish()

accuracy/train,▁
accuracy/valid,▁
oov/empty_sentences,▁
oov/occurrences,▁
oov/rate,▁
oov/tokens,▁
oov/unique_types,▁
accuracy/majority,0.50979
accuracy/test,0.51567
accuracy/train,0.5489
accuracy/valid,0.527
